# MÓDULO 3
## Tema 3. Métodos especiales y decoradores en clases

**Objetivos:**
- Usar métodos especiales (*dunder*) para que tus objetos sean “pythonicos”.
- Aplicar decoradores en clases: `@property`, `@classmethod`, `@staticmethod`.
- Reconocer casos típicos: representación, comparación, contenedores y constructores alternativos.

## Índice
- Métodos especiales (repr/str, eq, len/iter/contains)
- Decoradores en clases (`property`, `classmethod`, `staticmethod`)
- Notas sobre `dataclasses` (visión moderna)
- Trampas comunes y buenas prácticas

## Métodos especiales: qué son

Los métodos especiales hacen que tus clases se integren con el lenguaje:

- `print(obj)` → `__str__`
- `repr(obj)` → `__repr__`
- `obj1 == obj2` → `__eq__`
- `len(obj)` → `__len__`
- `for x in obj` → `__iter__`
- `x in obj` → `__contains__`

### `__repr__` vs `__str__`

- `__repr__`: útil para depurar; idealmente recreable.
- `__str__`: para mostrar a usuarios.

In [8]:
class Viaje:
    def __init__(self, origen, destino, precio):
        self.origen = origen
        self.destino = destino
        self.precio = float(precio)
    
v = Viaje(origen="Madrid", destino="Barcelona", precio=39.9)
print("print(v):", v)
print("repr(v):", repr(v))

print(v): <__main__.Viaje object at 0x000001BBB09D8980>
repr(v): <__main__.Viaje object at 0x000001BBB09D8980>


In [4]:
class Viaje:
    def __init__(self, origen, destino, precio):
        self.origen = origen
        self.destino = destino
        self.precio = float(precio)

    def __repr__(self):
        return (
            f"Viaje(origen={self.origen!r}, " # !r para ver internamente los tipos de dato
            f"destino={self.destino!r}, "
            f"precio={self.precio!r})"
        )

    def __str__(self):
        return f"{self.origen} → {self.destino} ({self.precio:.2f}€)"
    
v = Viaje(origen="Madrid", destino="Barcelona", precio=39.9)
print("print(v):", v)
print("repr(v):", repr(v))


print(v): Madrid → Barcelona (39.90€)
repr(v): Viaje(origen='Madrid', destino='Barcelona', precio=39.9)


### Igualdad: `__eq__`

Permite definir igualdad semántica (no necesariamente identidad).

Si implementas `__eq__`, piensa en:
- si quieres objetos “hasheables” (sets/dicts) → puede requerir `__hash__`.

In [12]:
class Usuario:
    def __init__(self, email):
        self.email = str(email).strip().lower()

u1 = Usuario(email="A@Ejemplo.com")
u2 = Usuario(email="a@ejemplo.com")
print(u1 == u2) # Aqui estamos comparando objetos en memoria, siempre sera False

False


In [6]:
class Usuario:
    def __init__(self, email):
        self.email = str(email).strip().lower()

    def __eq__(self, other):
        return self.email == other.email

u1 = Usuario(email="A@Ejemplo.com")
u2 = Usuario(email="a@ejemplo.com")
print(u1 == u2)
# Aqui Python esta ejecutando u1.__eq__(u2) que es: "a@ejemplo.com" == "a@ejemplo.com" -> True

True


### Contenedores: `__len__`, `__iter__`, `__contains__`

In [15]:
class Bolsa:
    def __init__(self, *items):
        self._items = list(items)

bolsa_compra = Bolsa("manzana", "pera", "plátano")
print("len:", len(bolsa_compra))
print("pera" in bolsa_compra)
for x in bolsa_compra:
    print("item:", x)

TypeError: object of type 'Bolsa' has no len()

In [7]:
class Bolsa:
    def __init__(self, *items):
        self._items = list(items)

    def __len__(self):
        return len(self._items)

    def __iter__(self):
        return iter(self._items)

    def __contains__(self, item):
        return item in self._items

bolsa_compra = Bolsa("manzana", "pera", "plátano")
print("len:", len(bolsa_compra))
print("pera" in bolsa_compra)
for x in bolsa_compra:
    print("item:", x)

len: 3
True
item: manzana
item: pera
item: plátano


## Decoradores en clases

### `@property`

Te deja exponer un atributo como interfaz pública, con validación o cálculo. Vamos a ver un ejemplo sin @property y otro con @property

In [16]:
class Producto:
    def __init__(self, precio):
        self.precio = precio

    def precio_con_descuento(self):
        return self.precio * 0.9

p = Producto(precio=100)
print(p.precio_con_descuento())
p.precio = -50   # nadie lo impide
print(p.precio_con_descuento())

90.0
-45.0


In [18]:
class Producto:
    def __init__(self, precio):
        self._precio = precio   

    @property
    def precio(self):
        return self._precio

    @precio.setter
    def precio(self, value):
        if value <= 0:
            raise ValueError("El precio debe ser positivo")
        self._precio = value

    @property
    def precio_con_descuento(self):
        # ✔ no modifica nada
        # ✔ no recibe parámetros
        # ✔ es un cálculo puro
        # → se expone como atributo (@property)
        return self._precio * 0.9

    def aplicar_descuento(self, porcentaje):
        # ❌ modifica el estado
        # ❌ recibe parámetros
        # → se expone como método (())
        self.precio = self._precio * (1 - porcentaje / 100)


p = Producto(100)

print(p.precio_con_descuento)

p.aplicar_descuento(10)
print(p.precio)

p.precio = 120
print(p.precio_con_descuento)

p.precio = -50  # ValueError: El precio debe ser positivo


90.0
90.0
108.0


ValueError: El precio debe ser positivo

### `@classmethod` como constructor alternativo

In [20]:
class Fecha:
    def __init__(self, year, month, day):
        # Constructor principal
        # Se usa cuando los datos ya vienen separados
        self.year = int(year)
        self.month = int(month)
        self.day = int(day)

    @classmethod
    def desde_iso(cls, iso_str):
        # Constructor alternativo 1
        # Crea una Fecha a partir de un string "YYYY-MM-DD"
        y, m, d = iso_str.split("-")
        return cls(y, m, d)

    @classmethod
    def desde_timestamp(cls, ts):
        # Constructor alternativo 2
        # Crea una Fecha a partir de un timestamp UNIX (segundos)
        import datetime
        dt = datetime.datetime.fromtimestamp(ts)
        return cls(dt.year, dt.month, dt.day)

    @classmethod
    def hoy(cls):
        # Constructor alternativo 3
        # Crea una Fecha con la fecha actual
        import datetime
        hoy = datetime.date.today()
        return cls(hoy.year, hoy.month, hoy.day)

    def __str__(self):
        return f"{self.day:02d}/{self.month:02d}/{self.year:04d}"

# --- Creación de objetos usando distintas "puertas de entrada" ---

f1 = Fecha(2026, 1, 21)                 # Constructor normal
f2 = Fecha.desde_iso("2026-01-21")      # Desde string ISO
f3 = Fecha.desde_timestamp(1768953600)  # Desde timestamp UNIX
f4 = Fecha.hoy()                        # Fecha actual

# --- Todos son objetos Fecha ---
print(f1)
print(f2)
print(f3)
print(f4)


21/01/2026
21/01/2026
21/01/2026
22/01/2026


### `@staticmethod` para utilidades cercanas

In [21]:
class Texto:
    @staticmethod
    def limpiar(s):
        return " ".join(str(s).strip().split())

print(Texto.limpiar("  hola   mundo  "))

hola mundo


In [22]:
class Fecha:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @staticmethod
    def es_fecha_valida(year, month, day):
        if month < 1 or month > 12:
            return False
        if day < 1 or day > 31:
            return False
        return True

print(Fecha.es_fecha_valida(2026, 1, 21))   # True
print(Fecha.es_fecha_valida(2026, 13, 40))  # False

True
False


## Notas sobre `dataclasses` (visión moderna)

En muchos casos, si tu clase es principalmente “datos”, puedes simplificar con `dataclasses`.

- Genera `__init__`, `__repr__` y comparaciones de forma automática.
- Es especialmente útil para modelos “ligeros”.

(Esto no sustituye POO; es una herramienta del lenguaje.)

In [23]:
from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class Evento:
    """
    Modelo de datos ligero:
    - Guarda información
    - Es inmutable
    - Tiene __init__, __repr__ y __eq__ automáticos
    """
    nombre: str
    fecha: date
    ciudad: str

    @property
    def es_futuro(self):
        # Cálculo simple derivado de los datos
        return self.fecha > date.today()

e1 = Evento(nombre="Concierto", fecha=date(2026, 7, 10), ciudad="Madrid")
e2 = Evento(nombre="Concierto", fecha=date(2026, 7, 10), ciudad="Madrid")

print(e1)
print("¿Es futuro?", e1.es_futuro)
print("¿Son iguales?", e1 == e2)

# e1.ciudad = "Barcelona"  # 💥 inmutable

Evento(nombre='Concierto', fecha=datetime.date(2026, 7, 10), ciudad='Madrid')
¿Es futuro? True
¿Son iguales? True


## Trampas comunes y buenas prácticas

- Define `__repr__` siempre que tus objetos se usen en notebooks/logs.
- Evita `__str__` ambiguos si el objeto se serializa/depura.
- Si defines `__eq__`, evalúa coherencia con `__hash__` (y con inmutabilidad).
- Usa `@classmethod` para parsing y formatos alternativos (ISO, CSV, etc.).
- `@staticmethod` es útil, pero no abuses: si no tiene que ver con la clase, muévelo a un módulo.

### Mini-práctica (sin solución aquí)
- Implementa `Vector2D` con `__add__` y `__repr__`.
- Crea `Usuario.desde_nombre_apellidos(...)` que construya un email normalizado.